## Example: How to use the ilastik Python API for Pixel Classification

The first version of the ilastik API allows you to predict your data with a previously trained ilastik project directly from Python.

Used data courtesy of the Gerlich Lab

In [ ]:
from ilastik.experimental.api import PixelClassificationPipeline
import numpy
from xarray import DataArray
# Add your imports here, e.g. for loading and preprocessing data
import imageio.v3 as iio
from matplotlib import pyplot as plt

In [ ]:
project_file = "pc.ilp"
pipeline = PixelClassificationPipeline.from_ilp_file(project_file)

In [ ]:
# load the image you would like to process and wrap it in an xarray.DataArray,
# providing the appropriate dimension names
image = DataArray(iio.imread("2d_cells_apoptotic_1channel.png"), dims=("y", "x"))
prediction = pipeline.get_probabilities(image)
# show the foreground channel:
plt.imshow(prediction[..., 0])

### Saving the results

The prediction is an `xarray.DataArray` with named dimensions.
The exact dimensions depend on the trained project and input data.
For this 2D single-channel example the result has dimensions `(y, x, c)`,
where `c` holds the per-class probabilities (0–1).

Below are two ways to write results to disk.

In [ ]:
# Save the full probability map as a multi-channel TIFF.
# Depending on how the TIFF metadata is interpreted,
# Fiji/ImageJ may show this as a multi-channel stack.
iio.imwrite("probabilities.tiff", prediction.values)

In [ ]:
# Alternatively, save as a NumPy .npy file for quick reloading in Python.
numpy.save("probabilities.npy", prediction.values)

### Deriving a segmentation

Often you want a single label per pixel rather than a full probability map.
Taking the `argmax` across the channel axis assigns each pixel to its most
likely class, giving you a simple segmentation mask.

In [ ]:
# For this example the channel axis is last, so axis=-1 selects it.
segmentation = numpy.argmax(prediction.values, axis=-1).astype(numpy.uint8)
iio.imwrite("segmentation.tiff", segmentation)

# Quick visual check
plt.imshow(segmentation, cmap="tab10")
plt.title("Segmentation (argmax of probabilities)")
plt.colorbar(label="class")
plt.show()